In [1]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

In [2]:
# ---- data (Table 1: bps effect of +1pp debt/GDP on long-term rates) ----
# point = single estimate; low/high = range (Kinoshita only)
rows = [
    ("Laubach (2003)",            3.50, None, None),
    ("Engen & Hubbard (2004)",    3.30, None, None),
    ("Gale & Orszag (2004)",      4.90, None, None),
    ("Kinoshita (2006)",          None, 2.00, 5.30),
    ("Laubach (2009)",            4.00, None, None),
    ("Kumar & Baldacci (2010)",   5.28, None, None),
    ("Li & Wei (2013)",           5.80, None, None),
    ("Tedeschi (2019)",           4.21, None, None),
    ("Warshawsky & Mantus (2022)",4.50, None, None),
    ("Gust & Skaperdas (2024)",   3.50, None, None),
    ("Cotton (2024)",             4.30, None, None),
]
df = pd.DataFrame(rows, columns=["study", "point", "low", "high"])
# a single column for sorting (use point, or range midpoint for Kinoshita)
df["sortval"] = df["point"].fillna((df["low"] + df["high"]) / 2)

# ---- EO theme ----
styles = EcoStyles()
styles.register_and_enable_theme()          # match your usual call / theme name

order = alt.EncodingSortField(field="sortval", order="descending")

# range bar for Kinoshita (only rows with low/high)
rng = (
    alt.Chart(df)
    .transform_filter("isValid(datum.low)")
    .mark_rule(strokeWidth=2, opacity=0.6)
    .encode(
        y=alt.Y("study:N", sort=order, title=None),
        x=alt.X("low:Q", title="Basis points per +1pp debt/GDP",
                scale=alt.Scale(domain=[0, 6.5])),
        x2="high:Q",
    )
)

# point estimates
pts = (
    alt.Chart(df)
    .transform_filter("isValid(datum.point)")
    .mark_point(filled=True, size=90, opacity=0.9)
    .encode(
        y=alt.Y("study:N", sort=order, title=None),
        x=alt.X("point:Q", title="Basis points per +1pp debt/GDP",
                scale=alt.Scale(domain=[0, 6.5])),
        tooltip=[alt.Tooltip("study:N", title="Study"),
                 alt.Tooltip("point:Q", title="bps", format=".2f")],
    )
)

chart = (rng + pts).properties(
    width=560, height=380,
    title=alt.Title(
        "Estimates cluster: debt raises long-term rates by roughly 3-5 bps",
        subtitle="Effect of a 1 percentage-point rise in debt/GDP on long-term interest rates (bps)",
    ),
)

styles.save(chart, name="interest_rate_dotplot",
            source="See Appendix Table 1 for studies and samples", svg=True)
chart

alt.LayerChart(...)